# Continued Training of EmberX Model (YOLOv8-OBB)

This notebook allows you to continue training the `emberx_v1.pt` model on Google Colab using a GPU (which is much faster than training locally on a CPU).

### Instructions:
1. **Set runtime type to GPU**: Click on **Runtime** -> **Change runtime type** -> select **T4 GPU** (or any available GPU) and click Save.
2. **Upload files**: Upload the following files to the Colab files sidebar (left panel):
   - `emberx_v1.pt` (your current model weights)
   - `Fire Smoke and Human Detector.v32i.yolov8-obb.zip` (the dataset zip file)
3. Run the cells below sequentially.

In [ ]:
# 1. Install ultralytics
!pip install ultralytics -q
from ultralytics import YOLO
import os
import ultralytics
ultralytics.checks()

### 2. Extract the Dataset
This cell extracts the dataset from your uploaded zip file.

In [ ]:
import zipfile

zip_path = 'Fire Smoke and Human Detector.v32i.yolov8-obb.zip'
extract_path = '/content/dataset/'

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f'Dataset extracted successfully to {extract_path}')
else:
    print(f'Error: {zip_path} not found. Please upload it to the Colab files panel.')

### 3. Update paths in `data.yaml`
Set the paths in `data.yaml` to match Colab's absolute paths.

In [ ]:
import yaml

yaml_path = '/content/dataset/data.yaml'
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)
    
    # Set paths matching Colab's extracted directory
    data_cfg['path'] = '/content/dataset'
    data_cfg['train'] = 'train/images'
    data_cfg['val'] = 'valid/images'
    data_cfg['test'] = 'test/images'
    
    with open(yaml_path, 'w') as f:
        yaml.dump(data_cfg, f)
    print('Updated data.yaml structure:')
    print(data_cfg)
else:
    print(f'Error: {yaml_path} not found.')

### 4. Load weights & start training
We will load your existing `emberx_v1.pt` weights and train them for more epochs. If `emberx_v1.pt` is not uploaded, it falls back to the pretrained `yolov8s-obb.pt` weights.

In [ ]:
model_path = 'emberx_v1.pt'

if os.path.exists(model_path):
    print(f"Loading existing model weights: {model_path}")
    model = YOLO(model_path)
else:
    print(f"{model_path} not found. Starting from pretrained yolov8s-obb.pt weights.")
    model = YOLO('yolov8s-obb.pt')

# Define training parameters
# Change EPOCHS to your desired number of epochs (e.g. 50 or 100)
EPOCHS = 50

results = model.train(
    data=yaml_path,
    epochs=EPOCHS,
    imgsz=640,
    batch=16,
    name='yolov8_obb_fire_smoke_continued',
    device=0  # Use Colab GPU
)

### 5. Download the new best model weights
This cell will automatically find the best trained weights and download them to your computer.

In [ ]:
from google.colab import files
import glob

best_weights = sorted(glob.glob('/content/runs/obb/yolov8_obb_fire_smoke_continued*/weights/best.pt'))
if best_weights:
    print(f"Downloading best weights from: {best_weights[-1]}")
    files.download(best_weights[-1])
else:
    print("Could not find the best.pt weights. Please check the run folder path.")